In [ ]:
# Install ffmpeg if not present
!apt-get install -y ffmpeg

# Create a new directory for resampled wavs
!mkdir -p /kaggle/working/dataset_resampled/wav

# Bulk convert to 22050Hz, Mono, 16-bit
!find /kaggle/input/datasets/yashsamant25/telugu-female-english/english/wav -name "*.wav" -exec sh -c 'ffmpeg -i "$1" -ar 22050 -ac 1 -c:a pcm_s16le "/kaggle/working/dataset_resampled/wav/$(basename "$1")" -loglevel error -y' _ {} \;

# Note: You will need to copy your txt.done.data to /kaggle/working/dataset_resampled/ as well so the script finds the new root!
!cp /kaggle/input/datasets/yashsamant25/telugu-female-english/english/txt.done.data /kaggle/working/dataset_resampled/

In [ ]:
!cp /kaggle/input/datasets/yashsamant25/telugu-female-english/english/txt.done.data /kaggle/working/dataset_resampled/

In [ ]:
%%writefile train.py
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import shutil
import sys
import glob
import subprocess

# ==========================================
# 0. EXORCISE THE CACHE (MUST BE FIRST)
# ==========================================
for path in ["/kaggle/working/phoneme_cache", "/kaggle/working/output"]:
    if os.path.exists(path):
        print(f"🧹 Deleting old ghost cache at {path}...")
        shutil.rmtree(path)

# ==========================================
# 1. SETUP & DEPENDENCIES
# ==========================================
try:
    print("⚙️ [1/5] Checking Dependencies...")
    subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "numpy>=1.26.0,<2.0",
    "scipy<1.13",
    "transformers<4.43.0",
    "coqui-tts",
    "pysbd",
    "torchaudio==2.2.2"
])
    os.system("apt-get update -y && apt-get install -y espeak-ng libsndfile1-dev")
except Exception as e:
    print(f"❌ Setup Failed: {e}")
    sys.exit(1)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchaudio")

from trainer import Trainer, TrainerArgs
from TTS.tts.configs.shared_configs import BaseDatasetConfig, BaseAudioConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor
from TTS.utils.manage import ModelManager

# ==========================================
# 2. DATASET & FORMATTER
# ==========================================
def find_dataset():
    # 🔴 Explicitly points to the new resampled directory
    meta_path = "/kaggle/working/dataset_resampled/txt.done.data"
    
    if not os.path.exists(meta_path):
        print(f"❌ ERROR: Could not find resampled metadata at {meta_path}")
        print("Did you run the bash ffmpeg resampling script first?")
        sys.exit(1)
        
    root_path = os.path.dirname(meta_path)
    print(f"✅ Found RESAMPLED dataset at: {root_path}")
    return root_path, meta_path

ROOT_PATH, META_FILE = find_dataset()

def indic_formatter(root_path, meta_file, **kwargs):
    items = []
    with open(meta_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            content = line.strip("()").strip()
            parts = content.split(" ", 1)
            if len(parts) < 2: continue
            file_id = parts[0]
            text = parts[1].replace('"', '').strip()
            wav_file = os.path.join(root_path, "wav", f"{file_id}.wav")
            if os.path.exists(wav_file):
                items.append({
                    "text": text,
                    "audio_file": wav_file,
                    "speaker_name": "ljspeech", 
                    "language": "en-us",
                    "root_path": root_path
                })
    return items

dataset_config = BaseDatasetConfig(
    formatter="indic_formatter", 
    meta_file_train=META_FILE,
    path=ROOT_PATH,
    language="en-us"
)

# ==========================================
# 3. CONFIGURATION (OPTIMIZED FOR SPEED)
# ==========================================
config = VitsConfig(
    run_name="vits_telugu_female_resampled", 
    batch_size=6,          # ⚡ Optimized for Kaggle GPUs
    eval_batch_size=3,      
    batch_group_size=3,     
    num_loader_workers=2,   # ⚡ Optimized for Kaggle CPUs
    mixed_precision=True,
    epochs=1000,
    save_step=1000,         
    output_path="/kaggle/working/output",
    datasets=[dataset_config],
    
    # --- PHONEMES ---
    use_phonemes=True,
    phonemizer="espeak",
    phoneme_language="en-us",
    compute_input_seq_cache=True,
    phoneme_cache_path="/kaggle/working/phoneme_cache",
    text_cleaner="phoneme_cleaners",
    
    # --- AUDIO ---
    audio=BaseAudioConfig(
        sample_rate=22050,      
        resample=False,     # ⚡ DISABLED. Using pre-resampled dataset!
        win_length=1024,
        hop_length=256,
        num_mels=80,
        mel_fmin=0,
        mel_fmax=None
    ),
    
    use_speaker_embedding=False 
)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    os.makedirs("/kaggle/working/phoneme_cache", exist_ok=True)
    
    ap = AudioProcessor.init_from_config(config)
    tokenizer, config = TTSTokenizer.init_from_config(config)
    
    train_samples, eval_samples = load_tts_samples(
        dataset_config,
        eval_split=True,
        formatter=indic_formatter 
    )
    
    print("⬇️ Downloading Teacher Model (LJSpeech)...")
    manager = ModelManager()
    model_path, config_path, _ = manager.download_model("tts_models/en/ljspeech/vits")
    
    print("🔥 Starting Training...")
    model = Vits(config, ap, tokenizer, speaker_manager=None)
    
    trainer = Trainer(
        TrainerArgs(restore_path=model_path),
        config,
        output_path="/kaggle/working/output",
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )
    
    trainer.fit()

In [ ]:
!python train.py

In [ ]:
!ls /kaggle/input/datasets/yashsamant25/telugu-female-english/english